# Jour 1 · Nettoyer et rééchantillonner


## Objectifs

- repérer doublons, trous et valeurs absentes
- remettre les mesures sur une grille temporelle régulière
- distinguer rééchantillonnage, agrégation et interpolation

Les capteurs perdent des messages, envoient parfois deux fois la même mesure ou arrivent en retard. Avant tout modèle, il faut rendre ces défauts visibles.

## Quatre problèmes différents à ne pas confondre

| Problème | Exemple | Risque |
|---|---|---|
| Lignes désordonnées | 10 h 30 apparaît avant 10 h 15 | calculs temporels faux |
| Timestamp dupliqué | deux messages à 10 h 15 | compter deux fois ou garder la mauvaise version |
| Valeur absente | la ligne de 10 h 15 existe, mais la température vaut `NaN` | variable inutilisable pour cet instant |
| Message absent | aucune ligne à 10 h 30 | trou invisible tant que la grille n'est pas créée |

`isna()` détecte les cases vides, mais ne peut pas compter une ligne qui n'existe pas. Pour voir les messages absents, il faudra d'abord construire la grille des instants attendus.

Notre contrat de données indique qu'un message est attendu toutes les **15 minutes**. Sans cette information métier, un intervalle de 30 minutes pourrait être un trou… ou la fréquence normale du capteur.

![Quatre tableaux montrant lignes désordonnées, timestamp dupliqué, valeur absente et message absent](../assets/jour_01/02_quatre_problemes_qualite.png)

*Ces quatre défauts se ressemblent parfois dans un graphique, mais ils ne se détectent ni ne se corrigent de la même manière.*

## Étape 1 — remettre les messages dans l'ordre et dresser l'inventaire

On commence sans modifier les valeurs : lecture, conversion du timestamp, tri, puis comptage. C'est une règle générale de nettoyage : **mesurer le problème avant de le corriger** afin de pouvoir expliquer ce qui a changé.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt


from pathlib import Path


def find_project_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "datasets").is_dir():
            return candidate
    raise FileNotFoundError("Dossier datasets introuvable. Lancez Jupyter depuis le projet.")


ROOT = find_project_root()
DATA_DIR = ROOT / "datasets"


plt.style.use("seaborn-v0_8-whitegrid")
raw = pd.read_csv(DATA_DIR / "raw" / "iot_hvac_raw.csv")
raw["timestamp"] = pd.to_datetime(raw["timestamp"], utc=True)
raw = raw.sort_values("timestamp")

print("Doublons de timestamp :", raw["timestamp"].duplicated().sum())
print("\nValeurs absentes :")
print(raw.isna().sum())

## Étape 2 — observer les intervalles réellement reçus

Après suppression temporaire des timestamps dupliqués, `diff()` soustrait chaque timestamp au précédent :

```text
timestamp    diff avec la ligne précédente
10:00        NaT
10:15        15 minutes
10:30        15 minutes
11:00        30 minutes  ← au moins un message attendu manque
```

`value_counts()` compte ensuite combien de fois chaque intervalle apparaît. L'intervalle le plus fréquent aide à confirmer la fréquence habituelle, mais il ne remplace pas le contrat du capteur.

In [ ]:
unique_times = raw.drop_duplicates("timestamp")["timestamp"].sort_values()
intervals = unique_times.diff().value_counts().sort_index()
print("Intervalles les plus fréquents entre deux mesures :")
print(intervals.head())

## Étape 3 — décider quoi faire des doublons

Nous gardons ici la dernière copie d'un timestamp avec `keep="last"`. C'est une **décision**, pas une vérité universelle.

Dans un vrai système, on pourrait :

- garder le message ayant la date d'ingestion la plus récente ;
- garder celui dont le statut qualité est le meilleur ;
- agréger plusieurs capteurs ayant mesuré au même instant ;
- conserver toutes les versions pour audit.

Il faut donc connaître la clé logique des données. Ici, un seul `device_id` est présent : un timestamp devrait correspondre à une seule ligne.

## Étape 4 — matérialiser les messages absents

`resample("15min").asfreq()` construit toutes les cases attendues sans inventer de mesure :

```text
Avant                              Après création de la grille
10:00 → 21.2 °C                    10:00 → 21.2 °C
10:15 → 21.4 °C                    10:15 → 21.4 °C
10:45 → 21.8 °C                    10:30 → NaN
                                   10:45 → 21.8 °C
```

Le trou de 10 h 30 devient enfin visible. `asfreq()` ne le remplit pas : il ajoute seulement la ligne manquante sur la grille temporelle.

![Tableau avant et après création d'une grille temporelle régulière](../assets/jour_01/02_asfreq_grille_temporelle.png)

*La ligne de 10 h 30 est ajoutée avec `NaN` : le défaut devient mesurable sans inventer de valeur.*

In [ ]:
deduplicated = raw.drop_duplicates("timestamp", keep="last").set_index("timestamp")
regular = deduplicated.resample("15min").asfreq()

print("Lignes après dédoublonnage :", len(deduplicated))
print("Lignes sur la grille régulière :", len(regular))
print("Trous temporels matérialisés :", regular["device_id"].isna().sum())

## Étape 5 — interpoler avec prudence

Interpoler signifie estimer une valeur entre des mesures connues. Si la température vaut 20 °C à 10 h et 22 °C à 10 h 30, une interpolation linéaire proposera 21 °C à 10 h 15.

Cette estimation est raisonnable uniquement si :

- le signal évolue assez progressivement ;
- le trou est court ;
- aucune rupture importante ne s'est produite pendant le trou.

Elle est dangereuse pour un état marche/arrêt, un compteur cumulatif ou une longue panne de communication. Une ligne droite peut masquer précisément l'incident que l'on cherche à détecter.

Dans le code :

- `method="time"` tient compte de la distance entre les timestamps ;
- `limit=4` limite le remplissage à quelques pas consécutifs ;
- `ffill()` propage l'identifiant de l'équipement, pas une mesure physique.

Nous conservons aussi un masque `was_missing` afin de savoir quelles températures n'étaient pas réellement observées.

![Comparaison entre l'interpolation d'un petit trou et d'une longue absence](../assets/jour_01/02_interpolation_courte_longue.png)

*Une estimation courte peut être défendable ; une longue ligne droite peut masquer une rupture ou une panne de communication.*

In [ ]:
numeric_columns = [
    "temperature_c", "humidity_pct", "power_kw",
    "pressure_bar", "vibration_mm_s"
]
filled = regular.copy()
was_missing = regular[numeric_columns].isna()
filled["device_id"] = filled["device_id"].ffill().bfill()
filled[numeric_columns] = filled[numeric_columns].interpolate(
    method="time", limit=4, limit_direction="both"
)

print("Valeurs encore absentes après interpolation limitée :")
print(filled.isna().sum())
imputed = was_missing & filled[numeric_columns].notna()
print("\nValeurs numériques estimées par interpolation :", int(imputed.sum().sum()))

## Étape 6 — changer de granularité avec une agrégation

Passer de 15 minutes à 1 heure rassemble quatre créneaux dans un seul. Il faut choisir comment les résumer selon leur sens physique :

| Type de signal | Agrégation fréquente | Question à laquelle elle répond |
|---|---|---|
| Température / pression | moyenne | quel était le niveau moyen pendant l'heure ? |
| Vibration / latence | maximum | quel a été le pire pic ? |
| Énergie consommée par intervalle | somme | quelle quantité totale a été consommée ? |
| État ou configuration | dernière valeur | quel était l'état en fin d'heure ? |

`resample("1h")` crée les groupes horaires. `agg({...})` indique ensuite la règle propre à chaque colonne. Le choix `mean`, `max`, `sum` ou `last` est une décision métier, pas seulement une décision de code.

![Mesures de vibration agrégées par moyenne puis par maximum](../assets/jour_01/02_moyenne_ou_maximum.png)

*La moyenne décrit le niveau général ; le maximum conserve le pic bref. Le bon choix dépend de la question métier.*

In [ ]:
hourly = filled.resample("1h").agg({
    "temperature_c": "mean",
    "humidity_pct": "mean",
    "power_kw": "mean",
    "pressure_bar": "mean",
    "vibration_mm_s": "max",
})
hourly.head()

### À vous de jouer — comparer deux agrégations

Calculez la vibration horaire avec la moyenne puis avec le maximum. Tracez les deux résultats sur les sept derniers jours et expliquez lequel conserve le mieux les pics.

**Indice :** resample('1h').agg(['mean', 'max']) calcule les deux séries.

In [ ]:
# Étape 1 : regroupez vibration_mm_s par heure.
# vibration_hourly = filled["vibration_mm_s"].resample("1h").agg([...])
# Étape 2 : gardez les sept derniers jours puis utilisez .plot(...).
pass

### À vous de jouer — mesurer le plus grand trou

À partir des timestamps uniques du fichier brut, trouvez le plus grand intervalle sans mesure.

**Indice :** La méthode diff() produit les écarts entre timestamps consécutifs.

In [ ]:
# gaps = unique_times.____().dropna()
# print(gaps.max())
pass

### À vous de jouer — choisir une stratégie avant de coder

Pour chaque situation, choisissez entre **ne pas remplir**, **interpoler**, **propager la dernière valeur**, **agréger par moyenne**, **agréger par maximum** ou **agréger par somme**. Justifiez en une phrase.

1. Deux mesures de température manquent entre deux valeurs stables.
2. Six heures complètes de vibration manquent pendant une perte réseau.
3. On veut savoir si un seuil de vibration a été dépassé pendant chaque heure.
4. Une colonne contient l'énergie consommée pendant chaque tranche de 15 minutes.

In [ ]:
# Notez vos quatre décisions et leurs justifications ici.
pass

## À retenir

- Nettoyer ne signifie pas remplir silencieusement tous les trous.
- Le choix de `mean`, `max`, `sum` ou `last` dépend du sens physique du signal.
- Il faut conserver une trace des données réellement mesurées et des valeurs reconstruites.

### Mini-mémo

| Besoin | Commande |
|---|---|
| Mesurer les écarts entre dates | `timestamps.diff()` |
| Supprimer les doublons | `drop_duplicates(...)` |
| Créer une grille régulière | `resample(...).asfreq()` |
| Estimer de petits trous | `interpolate(...)` |
| Propager une valeur connue | `ffill()` |
| Regrouper et résumer | `resample(...).agg(...)` |